인공신경망 훈련
scikit-learn을 이용해서 모델 개발하기

사용할 데이터: qualitative classification과 positive reaction을 통합해서 정의한 label 데이터 (descriptor까지 계산된 파일로 파일 업로드 했음)

1) descriptor 중 standard deviation이 낮은 descriptor 제거 (0.01 미만)

2) select K best를 이용해서 모델에 사용할 feature 선택. https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html

#제일 빨라서 select K best 사용. 다 뽑아서 모델 만들어 보는 등 여러 방식 selection 있음

3) MLP classifier에서 조정해야 할 hyperparameter 개수 체크.

1. 업로드한 데이터로 모델을 만든다. (제목: skin_irritation_2Ddesc.csv)
2. Featue selection (Select K best) 이거 어떻게 작동하는지? (Feature 선택은 3~5) (첫번째 hidden node 개수는 feature 개수를 넘지 않게. 차라리 layer를 늘려라)
3. scikit-learn MLP(multi layer perceptron)에서 조정해야 되는 hyperparameter는 어떤것들이 있는지?
4. for MLP hyperparameter: -> MLP 모델 여러개 (각 hyperparameter 별로 성능이 어떻게 변하는지?)

1. 데이터 로드 및 표준편차(Standard Deviation) 필터링
* 목적: 계산 오류 원인 제거 및 예측 변별력이 없는 특성(Feature) 삭제.
* 상세 작동 원리:
    * pandas를 사용하여 skin_irritation_2Ddesc.csv 데이터 로드.
    * 인공신경망은 문자열, 결측치(NaN), 무한대(inf) 값을 연산할 수 없으므로 에러가 발생함. 따라서 학습에 불필요한 텍스트 및 오류 유발 열(Column)을 완전히 제거함.
    * 남은 디스크립터의 표준편차를 계산하여 0.01 미만인 특성 제거.
    * 표준편차가 0에 가깝다는 것은 모든 화합물이 해당 특성에 대해 동일한 수치를 가짐을 의미함. 이는 피부 자극성 여부를 분류하는 데 전혀 단서가 되지 못하므로 사전에 배제함.

In [2]:
import pandas as pd
import numpy as np

# 데이터 로드 및 정답(y) 분리
df = pd.read_csv('skin_irritation_2Ddesc.csv')
y = df['label']

# 문자열 컬럼 제외 및 결측치/무한대 포함 컬럼 완전 삭제
X_raw = df.drop(columns=['Chemical_Name', 'standardized_smi', 'label'], errors='ignore')
X_clean = X_raw.replace([np.inf, -np.inf], np.nan).dropna(axis=1)

# 표준편차 0.01 미만 특성 필터링
std_dev = X_clean.std()
X_filtered = X_clean.loc[:, std_dev >= 0.01]

print(f"필터링 후 디스크립터 개수: {X_filtered.shape[1]}개")

필터링 후 디스크립터 개수: 144개


2. Feature selection (SelectKBest) 작동 원리 및 적용
* 목적: 과적합(Overfitting) 방지 및 상관성이 높은 핵심 특성 추출.
* 상세 작동 원리:
    * SelectKBest는 타겟 변수(정답)를 예측하는 데 통계적으로 가장 유의미한 상위 k개의 특성만 선택하는 알고리즘임.
    * 분류 문제이므로 분산분석(ANOVA F-value) 함수인 f_classif를 사용하여 각 특성과 정답 간의 연관성 점수를 평가함.
    * 현재 인체 피부 자극성 데이터(Human patch test)의 화합물 개수(N수)가 수십 개로 매우 적음. 특성이 너무 많으면 모델이 훈련 데이터만 암기해버리므로, 가장 핵심적인 5개(k=5)의 특성만 남겨 모델 구조를 단순화함.

In [3]:
from sklearn.feature_selection import SelectKBest, f_classif

# SelectKBest 정의: 분류 문제용(f_classif), 상위 5개(k=5) 선택
selector = SelectKBest(score_func=f_classif, k=5)

# 필터링된 데이터(X_filtered)와 정답(y)을 넣어 학습 및 변환 수행
X_selected_array = selector.fit_transform(X_filtered, y)

# 선택된 특성명 확인 및 DataFrame 재구성
selected_feature_names = selector.get_feature_names_out()
X_selected = pd.DataFrame(X_selected_array, columns=selected_feature_names)

print(f"선택된 5개 특성: {list(selected_feature_names)}")

선택된 5개 특성: ['MinAbsEStateIndex', 'Chi1', 'MolMR', 'fr_C_O_noCOO', 'fr_ester']


3. MLP Classifier 주요 하이퍼파라미터(Hyperparameter) 분석
* 목적: 다층 퍼셉트론(MLP) 모델의 아키텍처 및 가중치(Weight) 최적화 방식 통제.
* 상세 파라미터:
    * hidden_layer_sizes: 은닉층의 개수와 층별 노드(뉴런) 수. 입력 특성이 5개이므로 첫 번째 은닉층의 노드도 인풋 개수와 동일하게 5개((5,))로 설정함.
    * activation: 은닉층 노드에 비선형성을 부여하는 활성화 함수. relu, tanh, logistic(sigmoid) 등을 적용 가능.
    * solver: 가중치 최적화 알고리즘. 데이터셋의 크기가 수천 개 이하로 작을 때는 기본값(adam)보다 lbfgs가 수렴 속도 및 성능 면에서 더 우수하므로 lbfgs 적용 필수.
    * alpha: L2 정규화(Regularization) 패널티. 가중치가 과도하게 커지는 것을 막아 과적합 방지.
    * max_iter: 가중치 최적화를 반복하는 최대 에폭(Epoch) 수. 충분한 수렴을 위해 1000 지정.

4. 하이퍼파라미터 조합별 모델 훈련 및 성능 평가 (Loop)
* 목적: 다양한 조건에서의 모델 예측력 비교 및 최적 설정 도출.
* 상세 작동 원리:
    * 인공신경망은 입력 변수의 스케일(단위) 차이에 극도로 민감하므로 StandardScaler를 통한 평균 0, 분산 1 변환(Scaling)이 필수적임.
    * 은닉층의 깊이, 활성화 함수, 규제 강도(alpha)를 달리한 파라미터 조합 딕셔너리를 구성함.
    * for 반복문을 통해 각 조합마다 MLPClassifier를 반복 생성 및 학습(fit).
    * 테스트 데이터셋에 대한 예측 정확도(Accuracy)를 산출하여 어떤 구조에서 독성(자극성) 분류 성능이 극대화되는지 데이터프레임 형태로 비교.

1. 추가로 조정 가능한 하이퍼파라미터 (solver='lbfgs' 기준)
* tol (Tolerance, 기본값: 0.0001): 최적화 허용 오차. 오차 개선이 이 수치보다 낮으면 학습이 완료된 것으로 간주하고 조기 종료함. 이 값을 1e-5 등으로 더 낮추면, 목표 지점(Minimum)에 도달하기 위해 모델이 더 세밀하게 훈련함.
* max_fun (기본값: 15000): solver='lbfgs'일 때만 사용되는 고유 파라미터임. 손실 함수(Loss function)를 최대 몇 번까지 호출할 것인지 한계치를 설정함. 복잡한 구조에서 tol을 낮출 경우, 수렴하기까지 연산이 길어지므로 이 값을 20000 이상으로 늘려야 끝까지 학습이 보장됨.
* activation 추가 옵션: 'logistic' (시그모이드 함수). 기존 'relu', 'tanh' 외에 활용 가능함.

2. 사용이 불가능한(무시되는) 하이퍼파라미터들
* 배제 대상: learning_rate, batch_size, early_stopping, momentum, beta_1, beta_2, n_iter_no_change 등.
* 배제 이유: 이 파라미터들은 데이터가 수천 개 이상일 때 쓰는 확률적 경사 하강법(solver='sgd')이나 adam 최적화 알고리즘에서만 작동하도록 설계됨. solver='lbfgs'를 사용하는 현재 실습에서는 값을 입력해도 모델이 이를 무시함.


1. MLPClassifier의 추가 하이퍼파라미터 전체 개요
공식 문서에 따르면, 이전에 고정해 두었던 옵션 외에도 다양한 파라미터를 조합할 수 있음.
activation (4종): 'identity'(선형), 'logistic'(시그모이드), 'tanh', 'relu'.
solver (3종): 'lbfgs'(소규모 데이터 최적화), 'sgd'(확률적 경사 하강법), 'adam'(대규모 데이터 범용).
learning_rate (3종): 'constant', 'invscaling', 'adaptive'. (단, solver='sgd'일 때만 작동함).
momentum: 경사 하강법 업데이트를 가속하는 수치 (단, solver='sgd'일 때만 작동함).
tol: 훈련을 조기 종료하기 위한 정밀도 기준 값.
작동 조건 주의: learning_rate나 momentum 같은 파라미터는 lbfgs 최적화기에서는 완전히 무시됨. 모든 파라미터의 영향을 테스트하려면 solver에 sgd와 adam까지 모두 포함시켜야 함.

--------------------------------------------------------------------------------
2. ParameterGrid를 이용한 무제한 조합 생성 전략
목적: 일일이 파라미터 리스트를 작성하는 대신, sklearn.model_selection.ParameterGrid 모듈
을 사용하여 입력한 모든 파라미터의 교차 조합(수백~수천 개)을 자동으로 생성함.
결과 도출: 생성된 수백 개의 모델 중 정확도(Accuracy)와 모델 신뢰성(ROC AUC)이 가장 높게 나온 상위 10개의 최적 조합만 필터링하여 출력함.


In [4]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, recall_score, 
                             f1_score, matthews_corrcoef, roc_auc_score)
import pandas as pd
import warnings
warnings.filterwarnings('ignore') # 모델이 수렴하지 않을 때 발생하는 경고창 숨김 처리

# 1. 훈련/테스트 데이터 분리 및 정규화 (X_selected, y는 이전 단계 데이터)
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. 가능한 모든 하이퍼파라미터 그리드(Grid) 정의
# 공식 문서에 있는 핵심 파라미터를 모두 투입하여 280개 이상의 조합 생성
param_grid = {
    'hidden_layer_sizes': [(5,), (4, 2)],                    # 1층, 2층 구조 비교
    'activation': ['identity', 'logistic', 'tanh', 'relu'],  # 4가지 활성화 함수 모두 평가
    'solver': ['lbfgs', 'sgd', 'adam'],                      # 3가지 최적화 알고리즘 모두 평가
    'alpha': [0.0001, 0.01, 0.1],                            # 규제(패널티) 강도 3단계 
    'learning_rate': ['constant', 'adaptive'],               # sgd 전용: 학습률 변경 방식
    'momentum': [0.9, 0.99],                                 # sgd 전용: 모멘텀 수치 비교
    'tol': [1e-4, 1e-5],                                     # 정밀도 요구치 비교
    'max_iter': [1000]                                       # 수렴을 위해 반복 횟수 고정
}

grid = ParameterGrid(param_grid)
print(f"{len(grid)}개의 하이퍼파라미터 조합 테스트 시작")

clf_results = []

# 3. 수백 개의 하이퍼파라미터 조합을 자동으로 반복 훈련
for params in grid:
    mlp_clf = MLPClassifier(
        hidden_layer_sizes=params['hidden_layer_sizes'], 
        activation=params['activation'], 
        solver=params['solver'],
        alpha=params['alpha'],
        learning_rate=params['learning_rate'],
        momentum=params['momentum'],
        tol=params['tol'],
        max_iter=params['max_iter'], 
        random_state=42
    )
    
    try:
        # 모델 학습
        mlp_clf.fit(X_train_scaled, y_train)
        
        # 테스트 세트 예측
        y_pred = mlp_clf.predict(X_test_scaled)
        y_prob = mlp_clf.predict_proba(X_test_scaled)[:, 1]
        
        # 6가지 평가지표 계산
        cm = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
        
        acc = accuracy_score(y_test, y_pred)
        sens = recall_score(y_test, y_pred, zero_division=0)
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        f1 = f1_score(y_test, y_pred, zero_division=0)
        mcc = matthews_corrcoef(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_prob)
        
        # 결과 저장
        clf_results.append({
            "Solver": params['solver'],
            "Activation": params['activation'],
            "Hidden Layers": str(params['hidden_layer_sizes']),
            "Alpha": params['alpha'],
            "LR_Strategy": params['learning_rate'],
            "Momentum": params['momentum'],
            "Tol": params['tol'],
            "Accuracy": round(acc, 4),
            "Sensitivity": round(sens, 4),
            "Specificity": round(spec, 4),
            "F1-Score": round(f1, 4),
            "MCC": round(mcc, 4),
            "ROC AUC": round(roc_auc, 4)
        })
    except Exception:
        continue # 에러가 발생하는 비정상 조합은 스킵

# 4. 최종 성능 결과표 도출 (중복 제거 및 최상위 10개 추출)
results_df = pd.DataFrame(clf_results)

# lbfgs solver 등 특정 변수를 무시하는 알고리즘 때문에 발생하는 결과 중복을 제거함
results_df = results_df.drop_duplicates(
    subset=["Solver", "Activation", "Hidden Layers", "Alpha", "Tol", "Accuracy", "ROC AUC"]
)

# 분류 정확도(Accuracy) 및 모델 성능지표(ROC AUC) 기준 내림차순 정렬
results_df = results_df.sort_values(by=["Accuracy", "ROC AUC"], ascending=[False, False]).reset_index(drop=True)

print("\n[테스트 완료: 최상위 성능 하이퍼파라미터 조합 TOP 10]")
display(results_df.head(10))

576개의 하이퍼파라미터 조합 테스트 시작

[테스트 완료: 최상위 성능 하이퍼파라미터 조합 TOP 10]


,Solver,Activation,Hidden Layers,Alpha,LR_Strategy,Momentum,Tol,Accuracy,Sensitivity,Specificity,F1-Score,MCC,ROC AUC
0,lbfgs,identity,"(5,)",0.0001,constant,0.90,0.00010,0.875,0.5,1.0,0.6667,0.6547,1.0000
1,lbfgs,identity,"(5,)",0.0001,constant,0.90,0.00001,0.875,0.5,1.0,0.6667,0.6547,1.0000
2,lbfgs,identity,"(4, 2)",0.0001,constant,0.90,0.00010,0.875,0.5,1.0,0.6667,0.6547,1.0000
3,lbfgs,identity,"(4, 2)",0.0001,constant,0.90,0.00001,0.875,0.5,1.0,0.6667,0.6547,1.0000
4,lbfgs,identity,"(5,)",0.0100,constant,0.90,0.00010,0.875,0.5,1.0,0.6667,0.6547,1.0000
5,lbfgs,identity,"(5,)",0.0100,constant,0.90,0.00001,0.875,0.5,1.0,0.6667,0.6547,1.0000
6,lbfgs,identity,"(4, 2)",0.0100,constant,0.90,0.00010,0.875,0.5,1.0,0.6667,0.6547,1.0000
7,lbfgs,identity,"(4, 2)",0.0100,constant,0.90,0.00001,0.875,0.5,1.0,0.6667,0.6547,1.0000
8,sgd,tanh,"(5,)",0.0001,adaptive,0.99,0.00010,0.875,0.5,1.0,0.6667,0.6547,0.9167
9,sgd,tanh,"(5,)",0.0100,adaptive,0.99,0.00010,0.875,0.5,1.0,0.6667,0.6547,0.9167


In [5]:
results_df

,Solver,Activation,Hidden Layers,Alpha,LR_Strategy,Momentum,Tol,Accuracy,Sensitivity,Specificity,F1-Score,MCC,ROC AUC
0,lbfgs,identity,"(5,)",0.0001,constant,0.9,0.00010,0.875,0.5,1.0000,0.6667,0.6547,1.0
1,lbfgs,identity,"(5,)",0.0001,constant,0.9,0.00001,0.875,0.5,1.0000,0.6667,0.6547,1.0
2,lbfgs,identity,"(4, 2)",0.0001,constant,0.9,0.00010,0.875,0.5,1.0000,0.6667,0.6547,1.0
3,lbfgs,identity,"(4, 2)",0.0001,constant,0.9,0.00001,0.875,0.5,1.0000,0.6667,0.6547,1.0
4,lbfgs,identity,"(5,)",0.0100,constant,0.9,0.00010,0.875,0.5,1.0000,0.6667,0.6547,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,lbfgs,tanh,"(5,)",0.0100,constant,0.9,0.00001,0.500,0.0,0.6667,0.0000,-0.3333,0.5
177,lbfgs,tanh,"(5,)",0.1000,constant,0.9,0.00010,0.500,0.0,0.6667,0.0000,-0.3333,0.5
178,lbfgs,tanh,"(5,)",0.1000,constant,0.9,0.00001,0.500,0.0,0.6667,0.0000,-0.3333,0.5
179,lbfgs,relu,"(4, 2)",0.0001,constant,0.9,0.00010,0.500,0.5,0.5000,0.3333,0.0000,0.5


1. SelectKBest의 한계 (무조건 최선의 결과를 보장하지 않는 이유)
단일 변수 평가(Univariate)의 한계: SelectKBest는 타겟 변수(정답)와 개별 디스크립터 간의 1:1 통계적 연관성 점수(예: f_classif)만 독립적으로 평가하여 순위를 매김
.
특성 간 상호작용(Synergy) 누락: 단독으로는 변별력이 낮아 SelectKBest에서 버려진 변수라 할지라도, 다른 변수와 결합했을 때 피부 자극성을 완벽하게 분류하는 강력한 패턴을 형성할 수 있음. SelectKBest는 이러한 변수 간의 조합 시너지 효과를 완전히 무시하므로 최적의 결과를 보장하지 않음.

--------------------------------------------------------------------------------
2. 모든 디스크립터 조합 탐색의 현실적 제약 및 전략
조합 폭발 (Combinatorial Explosion): 표준편차 필터링을 거친 수십 개의 디스크립터로 가능한 모든 조합을 구하면 수억~수조 개(2 
n
 )의 모델을 훈련해야 하므로 연산 불가능 상태에 빠짐.
현실적 해결책 (2-Step Search):
SelectKBest로 통계적 유의성이 있는 상위 12개의 '1차 핵심 후보군'을 추림
.
파이썬의 itertools.combinations 모듈을 사용하여 1차 후보군 내에서 3~5개씩 뽑는 **모든 디스크립터 조합(약 1,500개)**을 직접 생성함.
이전 테스트에서 찾아낸 최적의 MLP 하이퍼파라미터 고정값에 이 1,500개의 디스크립터 조합을 번갈아 투입하여 실제 예측 성능(Accuracy, ROC AUC)을 랭킹화함.

--------------------------------------------------------------------------------
3. 디스크립터 모든 조합 교차 검증 통합 코드
설정 조건: lbfgs 최적화기, identity 활성화 함수 등 이전 단계에서 확인된 가장 우수한 하이퍼파라미터를 고정값으로 사용. 오로지 "어떤 디스크립터 조합이 들어갔을 때 성능이 극대화되는가"만 평가함.

In [11]:
import itertools
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# 1. 훈련/테스트 데이터 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X_filtered, y, test_size=0.2, random_state=42)

# 2. 1차 후보군 추출 (상위 12개 디스크립터 한정)
selector = SelectKBest(score_func=f_classif, k=12)
selector.fit(X_train, y_train)
candidate_features = X_filtered.columns[selector.get_support()]

# 3. 함께 변경하며 테스트할 하이퍼파라미터 조합 정의 (연산량 폭발 방지를 위해 핵심만 추림)
param_grid = {
    'hidden_layer_sizes': [(5,), (4, 2)],  # 1층, 2층 비교
    'activation': ['identity', 'tanh'],    # 선형, 비선형 최고 성능 비교
    'alpha': [0.0001, 0.01],               # 규제 강도 비교
    'solver': ['lbfgs'],                   # 소규모 데이터 필수
    'max_iter': [1000], 
    'random_state': [4]
}
grid = ParameterGrid(param_grid)

combo_results = []
print(f"🚀 총 {1507 * len(grid)}개의 모델(디스크립터 조합 × 하이퍼파라미터) 테스트를 시작합니다.")

# 4. [에러 수정 완료] 3개, 4개, 5개 디스크립터 조합 반복
for num_features in range(3, 6): 
    for feature_combo in itertools.combinations(candidate_features, num_features):
        feature_list = list(feature_combo)
        
        # 현재 조합 서브 데이터셋 생성 및 스케일링
        X_train_sub = X_train[feature_list]
        X_test_sub = X_test[feature_list]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_sub)
        X_test_scaled = scaler.transform(X_test_sub)
        
        # 5. 내부 루프: 해당 디스크립터 조합에 대해 다양한 하이퍼파라미터 적용
        for params in grid:
            mlp = MLPClassifier(**params)
            
            try:
                mlp.fit(X_train_scaled, y_train)
                
                y_pred = mlp.predict(X_test_scaled)
                y_prob = mlp.predict_proba(X_test_scaled)[:, 1]
                
                acc = accuracy_score(y_test, y_pred)
                roc_auc = roc_auc_score(y_test, y_prob)
                
                # 결과 기록 (디스크립터 정보와 모델 설정값 동시 저장)
                combo_results.append({
                    "Num Features": num_features,
                    "Feature Combination": ", ".join(feature_list),
                    "Hidden Layers": str(params['hidden_layer_sizes']),
                    "Activation": params['activation'],
                    "Alpha": params['alpha'],
                    "Accuracy": round(acc, 4),
                    "ROC AUC": round(roc_auc, 4)
                })
            except Exception:
                continue

# 6. 최종 랭킹 정렬 및 상위 결과 출력
results_df = pd.DataFrame(combo_results)

# 중복 결과 제거 및 성능순 정렬
results_df = results_df.drop_duplicates(
    subset=["Feature Combination", "Hidden Layers", "Activation", "Alpha"]
)
results_df = results_df.sort_values(by=["Accuracy", "ROC AUC"], ascending=[False, False]).reset_index(drop=True)

print("\n🏆 [디스크립터 및 하이퍼파라미터 통합 튜닝 완료: 최상위 성능 TOP 10]")
display(results_df.head(10))

🚀 총 12056개의 모델(디스크립터 조합 × 하이퍼파라미터) 테스트를 시작합니다.

🏆 [디스크립터 및 하이퍼파라미터 통합 튜닝 완료: 최상위 성능 TOP 10]


,Num Features,Feature Combination,Hidden Layers,Activation,Alpha,Accuracy,ROC AUC
0,3,"MinAbsEStateIndex, NumValenceElectrons, SlogP_...","(5,)",tanh,0.0100,1.0,1.0
1,3,"MinAbsEStateIndex, NumValenceElectrons, SlogP_...","(4, 2)",tanh,0.0100,1.0,1.0
2,3,"MinAbsEStateIndex, Chi0, SlogP_VSA6","(4, 2)",tanh,0.0001,1.0,1.0
3,3,"MinAbsEStateIndex, Chi0, SlogP_VSA6","(4, 2)",tanh,0.0100,1.0,1.0
4,3,"MinAbsEStateIndex, Chi0n, SlogP_VSA6","(4, 2)",tanh,0.0001,1.0,1.0
5,3,"MinAbsEStateIndex, Chi0n, SlogP_VSA6","(5,)",tanh,0.0100,1.0,1.0
6,3,"MinAbsEStateIndex, Chi1, SlogP_VSA6","(5,)",tanh,0.0001,1.0,1.0
7,3,"MinAbsEStateIndex, Chi1, SlogP_VSA6","(4, 2)",tanh,0.0001,1.0,1.0
8,3,"MinAbsEStateIndex, Chi1, SlogP_VSA6","(5,)",tanh,0.0100,1.0,1.0
9,3,"MinAbsEStateIndex, Chi1, SlogP_VSA6","(4, 2)",tanh,0.0100,1.0,1.0
